In [ ]:
!wget -qO- https://astral.sh/uv/install.sh | sh

!uv venv .venv --seed

!uv pip install -r requirements.txt

!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

In [15]:
import json
import os
import re
import sys
import random

from tqdm import tqdm
from pathlib import Path
from typing import Optional

import time
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

In [33]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS = 4096 # 32768

# os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

In [14]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [10]:
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

print("Model loaded.")

INFO 05-04 00:00:26 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.5, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-04 00:00:26 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-04 00:00:26 [model.py:1680] Using max model len 16384
INFO 05-04 00:00:26 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=2243) INFO 05-04 00:00:26 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None

(EngineCore pid=2243) Failed to get device capability: SM 12.x requires CUDA >= 12.9.
(EngineCore pid=2243) Failed to get device capability: SM 12.x requires CUDA >= 12.9.


(EngineCore pid=2243) INFO 05-04 00:00:31 [cuda.py:368] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=2243) INFO 05-04 00:00:31 [flash_attn.py:646] Using FlashAttention version 2
(EngineCore pid=2243) INFO 05-04 00:00:31 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...
(EngineCore pid=2243) INFO 05-04 00:00:31 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 1422.03 GiB.
(EngineCore pid=2243) INFO 05-04 00:00:31 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=2243) /workspace/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=2243)   torch._check_is_size(blocksize)


(EngineCore pid=2243) INFO 05-04 00:00:33 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 3.778699 seconds
(EngineCore pid=2243) INFO 05-04 00:00:37 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/279ff2fc43/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2243) INFO 05-04 00:00:37 [backends.py:1128] Dynamo bytecode transform time: 4.03 s
(EngineCore pid=2243) INFO 05-04 00:00:39 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 32768) from the cache, took 1.300 s
(EngineCore pid=2243) INFO 05-04 00:00:39 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/19b7976dfd380c01c4c594774f153791da26941be5053e4d99d9c191bfb391f4/rank_0_0/model
(EngineCore pid=2243) INFO 05-04 00:00:39 [monitor.py:53] torch.compile took 6.05 s in total
(EngineCore pid=2243) INFO 05-04 00:00:39 [monitor.py:81] Initial profiling/warmup run took 0.09 s
(EngineCore pid=

(EngineCore pid=2243) 2026-05-04 00:00:46,994 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2243) 2026-05-04 00:00:47,087 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136] EngineCore failed to start.
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136] Traceback (most recent call last):
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136]   File "/workspace/.venv/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136]     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136]                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136]   File "/workspace/.venv/lib/python3.11/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136]     return func(*args, **kwargs)
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py:1136]            ^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=2243) ERROR 05-04 00:00:47 [core.py

(EngineCore pid=2243) Process EngineCore:
(EngineCore pid=2243) Traceback (most recent call last):
(EngineCore pid=2243)   File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=2243)     self.run()
(EngineCore pid=2243)   File "/usr/lib/python3.11/multiprocessing/process.py", line 108, in run
(EngineCore pid=2243)     self._target(*self._args, **self._kwargs)
(EngineCore pid=2243)   File "/workspace/.venv/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 1140, in run_engine_core
(EngineCore pid=2243)     raise e
(EngineCore pid=2243)   File "/workspace/.venv/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=2243)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=2243)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=2243)   File "/workspace/.venv/lib/python3.11/site-packages/vllm/tracing/otel.py", line 178, in s

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [28]:
sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Sampling params loaded.")

Sampling params loaded.


In [29]:
# Build prompts for first 5 entries
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Rendering prompts:   0%|          | 0/1126 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1126 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


── Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even positive whole numbers are. The positive even whole numbers start at 2, right? So 2, 4, 6, 8, 10, and so on. So the first one is 2 (n=1), the second is 4 (n=2), the third is 6 (n=3), etc. So the nth positive even whole number is 2n. Let me confirm that:  ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s^2 + a^2) ds. Hmm, first, I need to check if this integral converges. Since it's an improper integral over the entire real line, we need to make sure the integral converges. The integrand is a^(3/2)/(s^2 + a^2). Let's note that a is a positive constant, probably, since we have ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) is about a turkey cooling down, so I think this

In [30]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 1126/1126 [00:46<00:00, 23.99it/s]

Scoring complete. 1126 results.


In [31]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  107 /  375  (28.53%)
  Free-form  :  369 /  751  (49.13%)
  Overall    :  476 / 1126  (42.27%)


In [32]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1126 records to results/starter_results.jsonl
